# 第六阶段：移动语义

## 实验 1：Value Categories

C++ 表达式不仅有 type，也有 value category。移动语义、引用绑定和重载选择都依赖表达式类别；只看变量声明中的类型并不足以解释行为。

本实验建立以下基础直觉：

- lvalue：具有 identity，可以稳定定位的表达式；
- prvalue：用于计算或初始化结果的纯值；
- xvalue：仍有 identity，但表示资源可以被复用的 expiring value；
- glvalue = lvalue + xvalue，rvalue = prvalue + xvalue；
- `std::move` 不执行移动，只把表达式转换为 xvalue。

运行方式：重启 `xcpp23` kernel 后从上到下执行所有单元，观察重载选择并核对编译期断言。

In [1]:
// 本步骤：引入断言、输出、字符串视图、类型检查和 std::move。
#include <cassert>
#include <iostream>
#include <string_view>
#include <type_traits>
#include <utility>

### 1. Value category 描述表达式，不描述对象

同一个对象可以通过不同类别的表达式出现。例如，命名变量 `value` 是 lvalue expression；`std::move(value)` 仍指向同一对象，却是 xvalue expression。

```text
                    expression
                        │
              ┌─────────┴─────────┐
              │                   │
            glvalue             rvalue
          has identity      can bind to T&&
              │                   │
         ┌────┴────┐         ┌────┴────┐
         │         │         │         │
       lvalue    xvalue    prvalue   xvalue
```

xvalue 同时属于 glvalue 和 rvalue：它既保留对象 identity，又允许选择移动操作。

### 2. 用引用重载观察绑定类别

重载可以直观显示表达式绑定到了 lvalue reference 还是 rvalue reference。注意 `int&&` 重载同时接受 prvalue 和 xvalue，因此它不能单独区分二者。

In [2]:
// 本步骤：定义三个引用重载，观察 mutable lvalue、const lvalue 和 rvalue 绑定。
std::string_view category(int &value) noexcept
{
    // 命名参数只用于形成重载，显式忽略避免未使用警告。
    static_cast<void>(value);
    return "mutable lvalue";
}

std::string_view category(const int &value) noexcept
{
    // const lvalue 不能绑定到可写 int&，因此选择只读引用。
    static_cast<void>(value);
    return "const lvalue";
}

std::string_view category(int &&value) noexcept
{
    // prvalue 和 xvalue 都能绑定到 rvalue reference。
    static_cast<void>(value);
    return "rvalue binding";
}

In [3]:
// 本步骤：把命名变量、const 变量、字面量和 std::move 结果送入重载。
{
    int value = 10;
    const int const_value = 20;

    // 有名字且可以稳定定位的 value 是 mutable lvalue expression。
    assert(category(value) == "mutable lvalue");

    // const_value 仍是 lvalue，但只能绑定到 const int&。
    assert(category(const_value) == "const lvalue");

    // 字面量 42 是 prvalue，绑定到 int&& 重载。
    assert(category(42) == "rvalue binding");

    // std::move(value) 是指向原对象的 xvalue，也绑定到 int&& 重载。
    assert(category(std::move(value)) == "rvalue binding");

    std::cout << "value: " << category(value) << '\n'
              << "const_value: " << category(const_value) << '\n'
              << "42: " << category(42) << '\n'
              << "move(value): " << category(std::move(value)) << '\n';
}

value: mutable lvalue
const_value: const lvalue
42: rvalue binding
move(value): rvalue binding


`&value` 合法说明 `value` 有可定位的 identity，但“能否直接取地址”只是帮助建立直觉，不是 value category 的完整定义。`&42` 不合法；`std::move(value)` 是 xvalue，虽然不能直接写 `&std::move(value)`，它仍表示原来那个有 identity 的对象。

### 3. 用 `decltype((expression))` 精确检查类别

对带额外括号的表达式使用 `decltype` 时：

- lvalue 得到 `T&`；
- xvalue 得到 `T&&`；
- prvalue 得到 `T`。

这里的额外括号很重要：`decltype(value)` 对未加括号的变量名执行特殊规则，直接给出声明类型 `int`；`decltype((value))` 才按表达式类别得到 `int&`。

In [4]:
// 本步骤：使用编译期类型断言精确区分 lvalue、xvalue 和 prvalue。
{
    int value = 10;

    // 命名变量表达式是 lvalue，因此 decltype((value)) 为 int&。
    static_assert(std::is_same_v<decltype((value)), int &>);

    // std::move(value) 产生 xvalue，因此结果为 int&&。
    static_assert(std::is_same_v<decltype((std::move(value))), int &&>);

    // 字面量是 prvalue，因此结果是非引用类型 int。
    static_assert(std::is_same_v<decltype((42)), int>);

    // 未加额外括号的变量名触发 decltype 特殊规则，返回声明类型 int。
    static_assert(std::is_same_v<decltype(value), int>);
}

### 4. 函数调用表达式的类别由返回类型决定

函数返回 `T&` 时，调用表达式是 lvalue；返回 `T&&` 时是 xvalue；返回普通 `T` 时是 prvalue。这也是 API 返回类型影响后续重载与生命周期的方式之一。

In [5]:
// 本步骤：定义分别返回值、lvalue reference 和 rvalue reference 的函数。
int make_value() noexcept
{
    // 按值返回，使调用表达式成为 prvalue。
    return 42;
}

int &borrow_value(int &value) noexcept
{
    // 返回现有对象的 lvalue reference，调用表达式仍是 lvalue。
    return value;
}

int &&expose_movable_value(int &value) noexcept
{
    // 返回指向现有对象的 rvalue reference，使调用表达式成为 xvalue。
    return std::move(value);
}

In [6]:
// 本步骤：验证三种返回类型对应的调用表达式类别和重载结果。
{
    int value = 10;

    // 普通值返回产生 prvalue，decltype 结果不是引用。
    static_assert(std::is_same_v<decltype((make_value())), int>);
    assert(category(make_value()) == "rvalue binding");

    // T& 返回产生 lvalue，并绑定到 mutable lvalue 重载。
    static_assert(std::is_same_v<decltype((borrow_value(value))), int &>);
    assert(category(borrow_value(value)) == "mutable lvalue");

    // T&& 返回产生 xvalue，并绑定到 rvalue reference 重载。
    static_assert(std::is_same_v<decltype((expose_movable_value(value))), int &&>);
    assert(category(expose_movable_value(value)) == "rvalue binding");
}

`expose_movable_value()` 返回的是对传入对象的引用，不拥有该对象。调用者必须保证原 `value` 仍然存活。value category 会影响如何使用表达式，但不会替代 ownership 和 lifetime contract。

### 5. 有名字的 rvalue reference 变量仍是 lvalue expression

变量的声明类型与使用该变量名时的表达式类别必须分开判断。`int&& reference` 的类型是 rvalue reference，但表达式 `reference` 有名字和 identity，因此是 lvalue。要再次允许移动，需要显式写 `std::move(reference)`。

In [7]:
// 本步骤：对比 rvalue reference 的声明类型与变量名表达式的类别。
{
    int &&reference = 42;

    // decltype(reference) 读取声明类型，因此得到 int&&。
    static_assert(std::is_same_v<decltype(reference), int &&>);

    // 加括号后按表达式分类：命名的 reference 是 lvalue。
    static_assert(std::is_same_v<decltype((reference)), int &>);
    assert(category(reference) == "mutable lvalue");

    // std::move 把命名 lvalue 再转换成 xvalue。
    assert(category(std::move(reference)) == "rvalue binding");
}

### 6. `std::move` 不会自动移动资源

`std::move(value)` 本质上是一次转换，表达“允许接收方把这个对象当作 expiring value”。只有随后选择到的移动构造、移动赋值或其他 `T&&` 重载，才可能真正转移资源。

```text
named object: value
        │ std::move：转换为 xvalue
        ▼
overload resolution
        │ 若目标类型实现 move operation
        ▼
resource transfer
```

本实验对 `int` 调用 `std::move` 并没有可转移资源。后续实验会用拥有动态数组的 `Buffer` 观察 copy 与 move 的真实差异。

### 7. 与 Native SDK 和 C ABI 的关系

value category 是 C++ 编译期表达式规则，不是可以写进 C ABI 的运行时标签。它帮助 C++ Core 选择复制、移动或借用接口；跨 ABI 后仍必须把语义翻译成 C 类型和明确 contract。

```text
C++ internal
lvalue / xvalue / prvalue
copy overload / move overload / borrow
                │
══════════ ABI Boundary ══════════
                │
C API
pointer / length / opaque handle
create / clone / transfer / destroy contract
```

Kotlin/Native 调用 C ABI 时不会看到 `T&&` 或 `std::move`。若 API 要转移 handle ownership，必须通过函数命名、文档、状态变化和配对销毁函数明确表达。

## 实验结论

| 表达式 | value category | 关键含义 |
| --- | --- | --- |
| 命名变量 `value` | lvalue | 有 identity，通常不会隐式选择移动 |
| const 命名变量 | lvalue | 有 identity，只能通过 const 借用读取 |
| 字面量 `42` | prvalue | 纯值，可绑定到 rvalue reference |
| `std::move(value)` | xvalue | 同一对象的 expiring 表达式，允许选择移动 |
| 命名的 `T&& reference` | lvalue | 类型是 rvalue reference，表达式本身仍有名字 |

需要形成的核心判断：

```text
expression = type + value category

std::move
→ 不移动对象
→ 产生 xvalue
→ 允许后续操作选择资源转移语义
```

下一实验将继续解释 `T&&`：为什么 rvalue reference 可以绑定临时值，以及为什么有名字的 rvalue reference 仍必须再次 `std::move`。